# 02 · Clasificador de titulares

**Pregunta:** ¿Puede un modelo pequeño aprender a clasificar un titular peruano en política, economía, deportes o ciencia/tecnología?

Etapa 2 (issue #2): etiquetar con un LLM y medir cuánto confiar en él.
Etapa 3 (issue #3): zero-shot, fine-tuning y evaluación. Las métricas se calculan **solo** con el conjunto de prueba.

In [1]:
import time
import unicodedata
from pathlib import Path

import pandas as pd
import anthropic
from dotenv import load_dotenv

# La clave vive en tarea_1/.env, nunca escrita en el notebook
load_dotenv()

True

## 1. Cargar los titulares

Los de `datos/noticias.csv`, recolectados en el notebook 01.

In [2]:
noticias = pd.read_csv("datos/noticias.csv")
noticias["bajada"] = noticias["bajada"].fillna("")
noticias.shape

(240, 7)

In [3]:
noticias["etiqueta"].value_counts()

etiqueta
deportes              93
politica              70
economia              40
ciencia_tecnologia    37
Name: count, dtype: int64

## 2. El prompt

El LLM recibe el titular y la bajada, y debe responder **una sola palabra** de una lista cerrada.
Como la respuesta puede variar un poco entre corridas, guardamos las etiquetas en un archivo y no las volvemos a pedir.

In [4]:
categorias_validas = ["politica", "economia", "deportes", "ciencia_tecnologia"]

instrucciones = (
    "Eres un editor de un diario peruano. Clasifica cada noticia en UNA de estas categorias:\n"
    "- politica: gobierno, Congreso, partidos, elecciones, justicia y autoridades\n"
    "- economia: precios, empleo, empresas, finanzas, comercio, dolar, inflacion\n"
    "- deportes: futbol y cualquier otro deporte, torneos, deportistas\n"
    "- ciencia_tecnologia: ciencia, tecnologia, salud, medio ambiente, innovacion\n"
    "- otra: si no encaja con claridad en ninguna de las cuatro\n"
    "Responde solo con la palabra de la categoria, sin tildes, sin explicacion."
)

cliente = anthropic.Anthropic()

In [5]:
respuesta = cliente.messages.create(
    model="claude-haiku-5-5",
    max_tokens=300,
    system=instrucciones,
    messages=[{"role": "user", "content": "Titular: Congreso aprueba nueva ley de pensiones"}],
)

# La respuesta es una lista de bloques; algunos son razonamiento. Buscamos el de texto.
for bloque in respuesta.content:
    print(bloque.type)

text


In [6]:
for bloque in respuesta.content:
    if bloque.type == "text":
        print(bloque.text)

politica


## 3. Clasificar un titular, y limpiar la respuesta

El modelo puede devolver `Política` con tilde y mayúscula. Antes de comparar normalizamos, y lo que no esté en la lista queda como `sin_clasificar`.

In [7]:
def normalizar(texto):
    # NFD separa la letra de su tilde; quitamos las marcas y dejamos la letra base
    separado = unicodedata.normalize("NFD", texto.lower().strip())
    return "".join(letra for letra in separado if unicodedata.category(letra) != "Mn")

In [8]:
normalizar("Política"), normalizar("  Ciencia_Tecnología ")

('politica', 'ciencia_tecnologia')

In [9]:
# Esta función se llama una vez por titular (cientos de veces), así que ahora sí conviene
def clasificar_titular(titular, bajada):
    texto = "Titular: " + titular + "\nBajada: " + bajada

    for intento in range(2):
        try:
            respuesta = cliente.messages.create(
                model="claude-haiku-5-5",
                max_tokens=300,
                system=instrucciones,
                messages=[{"role": "user", "content": texto}],
            )
            texto_respuesta = ""
            for bloque in respuesta.content:
                if bloque.type == "text":
                    texto_respuesta = bloque.text
            etiqueta = normalizar(texto_respuesta)
            if etiqueta in categorias_validas:
                return etiqueta
            return "otra"
        except anthropic.APIError:
            time.sleep(2)

    return "sin_clasificar"

In [10]:
clasificar_titular(noticias.loc[0, "titular"], noticias.loc[0, "bajada"])

'economia'

## 4. Etiquetar todos los titulares

Guardamos en `datos/etiquetas_llm.csv`. Si ya hay etiquetas de corridas anteriores, **no se repiten**: solo se paga por las notas nuevas.

In [11]:
ruta_etiquetas = Path("datos/etiquetas_llm.csv")

if ruta_etiquetas.exists():
    etiquetas_previas = pd.read_csv(ruta_etiquetas)
else:
    etiquetas_previas = pd.DataFrame(columns=["url", "etiqueta_llm"])

pendientes = noticias[~noticias["url"].isin(etiquetas_previas["url"])]
print("Ya etiquetados:", len(etiquetas_previas), "| Por etiquetar:", len(pendientes))

Ya etiquetados: 240 | Por etiquetar: 0


In [12]:
filas_nuevas = []

for posicion, fila in pendientes.iterrows():
    etiqueta_llm = clasificar_titular(fila["titular"], fila["bajada"])
    filas_nuevas.append({"url": fila["url"], "etiqueta_llm": etiqueta_llm})

etiquetas_nuevas = pd.DataFrame(filas_nuevas, columns=["url", "etiqueta_llm"])
len(etiquetas_nuevas)

0

In [13]:
etiquetas_llm = pd.concat([etiquetas_previas, etiquetas_nuevas], ignore_index=True)
etiquetas_llm.to_csv(ruta_etiquetas, index=False, encoding="utf-8")
etiquetas_llm["etiqueta_llm"].value_counts()

etiqueta_llm
deportes              92
politica              72
ciencia_tecnologia    37
economia              34
otra                   5
Name: count, dtype: int64

## 5. ¿Coincide el LLM con la sección del diario?

La etiqueta por sección (`etiqueta`) no es la verdad absoluta: una nota de la sección Política puede ser sobre economía. Pero sirve para detectar desacuerdos y revisarlos.

In [14]:
datos = noticias.merge(etiquetas_llm, on="url")
datos["coincide"] = datos["etiqueta"] == datos["etiqueta_llm"]
datos["coincide"].mean()

np.float64(0.9375)

In [15]:
pd.crosstab(datos["etiqueta"], datos["etiqueta_llm"])

etiqueta_llm,ciencia_tecnologia,deportes,economia,otra,politica
etiqueta,,,,,
ciencia_tecnologia,32,0,0,3,2
deportes,0,92,0,1,0
economia,4,0,33,1,2
politica,1,0,1,0,68


In [16]:
desacuerdos = datos[~datos["coincide"]]
desacuerdos[["titular", "seccion_original", "etiqueta", "etiqueta_llm"]].head(15)

,titular,seccion_original,etiqueta,etiqueta_llm
3,Papa León XIV Últimas Noticias: Sumo pontífice...,politica,politica,ciencia_tecnologia
5,Reinfo: formalización minera hasta 2028 exigir...,economia,economia,politica
32,Minería ilegal elevó 8 veces la demanda de com...,politica,politica,economia
36,Feriado por visita del papa León XIV: ministro...,economia,economia,politica
79,Ni en el refrigerador ni en el frutero: este e...,ciencia,ciencia_tecnologia,otra
120,Inccosac trae las primeras ‘tapas antirrobo’ p...,economia,economia,ciencia_tecnologia
122,La gran transformación del patrimonio: nuevas ...,economia,economia,otra
131,Apenas el 17% de los puntos críticos por el Fe...,economia,economia,ciencia_tecnologia
136,El fenómeno de El Niño y nuestros desafíos ene...,economia,economia,ciencia_tecnologia
141,La IA agnóstica,economia,economia,ciencia_tecnologia


## 6. Validación humana: 20 titulares a mano

El archivo de revisión **no incluye** la etiqueta del LLM ni la de la sección, para que tu juicio no quede influido.
La muestra se sortea una sola vez (`random_state=42`) y se reutiliza.

In [17]:
ruta_muestra = Path("datos/muestra_revision.csv")

if not ruta_muestra.exists():
    muestra = noticias.sample(20, random_state=42)
    muestra = muestra[["url", "titular", "bajada"]].copy()
    muestra["etiqueta_humana"] = ""
    muestra.to_csv(ruta_muestra, index=False, encoding="utf-8")

pd.read_csv(ruta_muestra)[["titular", "bajada", "etiqueta_humana"]]

,titular,bajada,etiqueta_humana
0,Chats de los asesinos de “China Polo”: “Si no ...,De terror. En el celular del sicario Luis Obis...,politica
1,Adenda de Cálidda para llevar gas a siete regi...,El principal punto en discusión es cómo se rec...,economia
2,Ministro César Astudillo tras incidente en la ...,"El ministro del Interior, César Astudillo, ase...",politica
3,Juntos por el Perú impulsa propuestas que figu...,"Más de cinco propuestas legislativas, que figu...",politica
4,"Revelan que diputada Graciela Chipana, de Junt...","La diputada Graciela Chipana, de Juntos por el...",politica
5,"Nekroos, actual campeón de Red Bull Batalla, l...",El encuentro se realizará el miércoles 7 de oc...,otra
6,Científicos descubren cómo predecir cuándo una...,Investigadores desarrollaron una fórmula matem...,ciencia_tecnologia
7,Congreso: ¿Hay consenso en el Senado para inco...,La Comisión de Constitución del Senado volverá...,politica
8,Una enorme cámara subterránea y natural fue de...,Una expedición en 1981 llegó a una cavidad que...,ciencia_tecnologia
9,¿Es sostenible la deuda de EE.UU.?,“Si el costo de financiar la deuda sigue aumen...,economia


**Tu turno:** abre `datos/muestra_revision.csv` y completa la columna `etiqueta_humana` con una de
`politica`, `economia`, `deportes`, `ciencia_tecnologia` u `otra`. Guarda y vuelve a correr la celda siguiente.

In [18]:
revision = pd.read_csv(ruta_muestra)
revision["etiqueta_humana"] = revision["etiqueta_humana"].fillna("").astype(str).str.strip()
revisadas = revision[revision["etiqueta_humana"] != ""]

if len(revisadas) < 20:
    print("Faltan etiquetas humanas:", 20 - len(revisadas), "de 20")
else:
    comparacion = revisadas.merge(etiquetas_llm, on="url")
    aciertos = (comparacion["etiqueta_humana"] == comparacion["etiqueta_llm"]).sum()
    print("Acuerdo LLM-humano:", aciertos, "de", len(comparacion))

Acuerdo LLM-humano: 20 de 20


## 7. Separar entrenamiento y prueba (issue #3)

In [19]:
# TODO: train_test_split con stratify y random_state fijo. Guardar el conjunto de prueba aparte.

## 8. Línea base: clasificación zero-shot

Sin entrenar nada: ¿cuánto acierta un modelo que nunca vio nuestros datos? Es el número a superar.

In [20]:
# TODO: pipeline("zero-shot-classification") sobre el conjunto de prueba -> F1 macro zero-shot

## 9. Fine-tuning de DistilBERT o BETO

In [21]:
# TODO: elegir modelo (p. ej. "dccuchile/bert-base-spanish-wwm-cased" o un DistilBERT multilingüe)
# TODO: tokenizar, entrenar pocas épocas, guardar en modelo/

## 10. Evaluación en el conjunto de prueba

In [22]:
# TODO: F1 macro, classification_report y matriz de confusión -- solo con prueba

## Qué hicimos

| Paso | Código | Resultado a reportar |
|---|---|---|
| Prompt con lista cerrada | `system=instrucciones` | |
| Limpiar la respuesta | `normalizar`, `categorias_validas` | |
| Etiquetar sin repetir | `isin`, `etiquetas_llm.csv` | |
| Comparar con la sección | `merge`, `crosstab` | coincidencia: ver sección 5 |
| Validación humana | `muestra_revision.csv` | acuerdo: __/20 |
| Zero-shot | | F1 macro: __ |
| Fine-tuning | | F1 macro: __ |